# DuckLake Conference Live Demo

Assumes the table already exists. Run `streaming_demo` first to populate it with
millions of rows. This notebook walks through: connect + explore, ACID transactions,
time travel, schema evolution + CDC, MERGE/upsert, and maintenance.

In [1]:
import sys
from pathlib import Path

from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

_REPO = Path.cwd()
if _REPO.name == "notebooks":
    _REPO = _REPO.parent
if str(_REPO / "src") not in sys.path:
    sys.path.insert(0, str(_REPO / "src"))

from ducklake_playground import DuckLakeEngine, load_config

config = load_config(_REPO / "config.yaml")
engine = DuckLakeEngine()
# Must match the storage backend used when `streaming_demo` created the table.
storage_mode = "local"  # or "s3"
engine.setup(config, storage_mode)
con = engine.connection
catalog = engine.catalog_name
TABLE = "demo_table"
fq = f"{catalog}.main.{TABLE}"
print(f"Connected to {catalog} | table: {fq}")

2026-05-21 21:50:55.464 | INFO     | ducklake_playground.engine:setup:141 - DuckLake engine setup complete (storage=local, data_path=/Users/graziano/GitHub/ducklake_playground/data/ducklake/, pg_baseline=9325.7 KB)


Connected to playground_ducklake_local | table: playground_ducklake_local.main.demo_table


## 1. Explore the Table

In [2]:
con.execute(f"DESCRIBE {fq}").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""id""","""BIGINT""","""YES""",null,"""NULL""",null
"""event_date""","""DATE""","""YES""",null,"""NULL""",null
"""int8_col""","""TINYINT""","""YES""",null,"""NULL""",null
"""int16_col""","""SMALLINT""","""YES""",null,"""NULL""",null
"""int32_col""","""INTEGER""","""YES""",null,"""NULL""",null
…,…,…,…,…,…
"""text_col""","""VARCHAR""","""YES""",null,"""NULL""",null
"""bool_col""","""BOOLEAN""","""YES""",null,"""NULL""",null
"""list_col""","""INTEGER[]""","""YES""",null,null,null


In [3]:
con.execute(f"""
    SELECT COUNT(*)                   AS total_rows,
           MIN(event_date)            AS first_date,
           MAX(event_date)            AS last_date,
           COUNT(DISTINCT event_date) AS partitions
    FROM {fq}
""").pl()

total_rows,first_date,last_date,partitions
i64,date,date,i64
1000000,2024-01-01,2024-01-30,30


In [4]:
con.execute(f"""
    SELECT varchar_col,
           COUNT(*)         AS cnt,
           SUM(int64_col)   AS total,
           AVG(float64_col) AS avg_val
    FROM {fq}
    WHERE event_date BETWEEN DATE '2024-01-10' AND DATE '2024-01-15'
    GROUP BY varchar_col
    ORDER BY cnt DESC
    LIMIT 10
""").pl()

varchar_col,cnt,total,avg_val
str,i64,"decimal[38,0]",f64
"""value_529""",249,-151148723910259724750,-1.9045e13
"""value_305""",243,-30540907389986659864,9.2362e13
"""value_800""",239,63740955580357578695,-1.4480e13
"""value_922""",239,85405002404505944155,5.4239e12
"""value_686""",238,58514410144643346574,-1.5752e13
"""value_313""",234,-32816697197540210426,-1.0947e13
"""value_174""",234,-82304733355348137510,-6.4248e12
"""value_818""",233,-97668297955158875317,4.6219e13
"""value_252""",232,10238548069084534080,-1.5680e13


In [5]:
# Prove partition pruning to the audience
print(con.execute(f"""
    EXPLAIN ANALYZE
    SELECT varchar_col, COUNT(*) AS cnt
    FROM {fq}
    WHERE event_date = DATE '2024-01-15'
    GROUP BY varchar_col
""").pl()["explain_value"].first())

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE     SELECT varchar_col, COUNT(*) AS cnt     FROM playground_ducklake_local.main.demo_table     WHERE event_date = DATE '2024-01-15'     GROUP BY varchar_col 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0201s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ─

## 2. ACID Transactions

Multi-statement transaction: INSERT + UPDATE land atomically in one DuckLake snapshot.

In [6]:
pre_count = con.execute(f"SELECT COUNT(*) FROM {fq}").fetchone()[0]
pre_snapshot = con.execute(
    f"SELECT MAX(snapshot_id) FROM {catalog}.snapshots()"
).fetchone()[0]
print(f"Before transaction: {pre_count:,} rows | snapshot v{pre_snapshot}")

Before transaction: 1,000,000 rows | snapshot v29


In [7]:
con.execute("BEGIN TRANSACTION")
try:
    # Insert two new rows
    con.execute(f"""
        INSERT INTO {fq} (id, event_date, int64_col, float64_col, varchar_col)
        VALUES
            (900000001, DATE '2024-01-15', 1499, 99.95, 'value_042'),
            (900000002, DATE '2024-01-15', 49,   19.99, 'value_007')
    """)
    # Update one of them (10% discount)
    con.execute(f"""
        UPDATE {fq}
        SET float64_col = float64_col * 0.9
        WHERE id = 900000001
    """)
    con.execute("COMMIT")
    print("COMMITTED")
except Exception as exc:
    con.execute("ROLLBACK")
    print(f"ROLLED BACK: {exc}")

post_count = con.execute(f"SELECT COUNT(*) FROM {fq}").fetchone()[0]
post_snapshot = con.execute(
    f"SELECT MAX(snapshot_id) FROM {catalog}.snapshots()"
).fetchone()[0]
print(f"After transaction: {post_count:,} rows (v{post_snapshot}, +{post_count - pre_count})")

COMMITTED
After transaction: 1,000,002 rows (v30, +2)


In [8]:
# Verify: both the INSERT and UPDATE landed
con.execute(f"""
    SELECT id, event_date, int64_col, float64_col, varchar_col
    FROM {fq}
    WHERE id IN (900000001, 900000002)
    ORDER BY id
""").pl()

id,event_date,int64_col,float64_col,varchar_col
i64,date,i64,f64,str
900000001,2024-01-15,1499,89.955,"""value_042"""
900000002,2024-01-15,49,19.99,"""value_007"""


## 3. Time Travel & Snapshots

Each write creates a new immutable snapshot. Query any historical version.

In [9]:
con.execute(f"""
    SELECT snapshot_id, snapshot_time, changes
    FROM {catalog}.snapshots()
    ORDER BY snapshot_id DESC
    LIMIT 10
""").pl()

snapshot_id,snapshot_time,changes
i64,"datetime[μs, Europe/Rome]",list[struct[2]]
30,2026-05-21 21:52:42.016338 CEST,"[{""inlined_insert"",[""9""]}]"
29,2026-05-21 21:50:19.992743 CEST,"[{""tables_inserted_into"",[""9""]}]"
28,2026-05-21 21:50:19.858282 CEST,"[{""tables_altered"",[""9""]}]"
27,2026-05-21 21:50:19.823827 CEST,"[{""tables_created"",[""main.demo_table""]}]"
26,2026-05-21 21:50:19.257439 CEST,"[{""tables_dropped"",[""7""]}]"


In [10]:
print(f"Before-tx snapshot: v{pre_snapshot} | After-tx snapshot: v{post_snapshot}")

Before-tx snapshot: v29 | After-tx snapshot: v30


In [11]:
# Query the table AS OF the previous snapshot (before our transaction)
con.execute(f"""
    SELECT COUNT(*) AS row_count_before_tx
    FROM {fq} AT (VERSION => {pre_snapshot})
""").pl()

row_count_before_tx
i64
1000000


In [12]:
# Prove the inserted rows did NOT exist in the previous version
con.execute(f"""
    SELECT id, event_date, float64_col
    FROM {fq} AT (VERSION => {pre_snapshot})
    WHERE id IN (900000001, 900000002)
""").pl()

id,event_date,float64_col
i64,date,f64


## 4. Change Data Feed (CDC)

What changed between two snapshots? No Kafka or external tooling required.

In [13]:
con.execute(f"""
    SELECT *
    FROM {catalog}.table_changes('{TABLE}', {pre_snapshot}, {post_snapshot})
    ORDER BY change_type, id
    LIMIT 20
""").pl()

snapshot_id,rowid,change_type,id,event_date,int8_col,int16_col,int32_col,int64_col,float32_col,float64_col,decimal_col,datetime_col,timestamp_col,varchar_col,text_col,bool_col,list_col,struct_col,map_col
i64,i64,str,i64,date,i8,i16,i32,i64,f32,f64,"decimal[18,4]",datetime[μs],"datetime[μs, Europe/Rome]",str,str,bool,list[i32],struct[3],list[struct[2]]
29,0,"""insert""",0,2024-01-01,-43,-15990,1980400120,8826520244630622010,374214.4375,-7.9268e14,-71384230759064.8438,2023-10-27 16:51:14.128559,2021-03-02 01:04:20.620557 CET,"""value_591""","""ar053rlzx88mhfktzze8r7978yw96v…",false,[-302807056],"{-645562509,""s_073"",-400996.824206}","[{""key_32"",-92}, {""key_44"",-771}]"
29,1,"""insert""",1,2024-01-01,81,29819,494109348,2432258467393470515,547741.1875,-8.4194e14,36274475894563.9141,2024-08-23 03:26:42.979445,2024-05-22 08:20:31.769028 CEST,"""value_270""","""b7lb2bmr2cixyvt9s75vwn7g96zc1w…",false,"[-1184971712, -1577161268, … -373532543]","{-55700468,""s_087"",-349562.621714}","[{""key_35"",-620}, {""key_40"",788}, … {""key_13"",88}]"
29,2,"""insert""",2,2024-01-01,-44,21,787767063,9205541960314996748,109943.054688,-7.4573e14,-37910897367114.8594,2023-09-26 18:29:23.733920,2021-04-14 01:41:30.936140 CEST,"""value_947""","""uupy835c3zdim4d73g9ye3r3apu4pe…",true,"[1181380818, 355486939, … 743780795]","{-305780694,""s_023"",-621908.052507}","[{""key_30"",-54}, {""key_24"",-305}, … {""key_06"",-216}]"
29,3,"""insert""",3,2024-01-01,-2,27511,2065725553,2016286853360654461,965441.4375,-3.1480e14,33998809206632.0078,2022-08-02 09:36:16.900246,2024-05-21 23:57:28.660077 CEST,"""value_260""","""gwiji1d8rnavubbdzatghcaxmzv8zt…",false,"[1281624326, 621008709, … -607232289]","{822753089,""s_061"",-493509.864803}","[{""key_37"",-348}, {""key_08"",-909}, … {""key_15"",-5}]"
29,4,"""insert""",4,2024-01-01,62,28716,-1269428629,4855144733824420640,348720.0,-3.5040e14,-63760936706371.5547,2024-08-19 21:34:12.783479,2022-11-17 19:47:49.535439 CET,"""value_652""","""0ykkaak1ovni91qkc6p0ule0fw8fxp…",true,"[-817406571, 459253678, … 733918036]","{1145202011,""s_075"",-57099.493971}","[{""key_42"",-428}]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
29,15,"""insert""",15,2024-01-01,104,7751,449971772,-1955114574302209106,-753963.6875,-2.6595e14,78981481386922.5156,2020-11-03 01:49:33.556885,2020-09-09 10:28:02.799656 CEST,"""value_231""","""nzrtpv3503h9g1mndfq45vxtx3poiy…",true,"[427729889, 1948078299, … -507858022]","{-721236004,""s_006"",-689967.093792}","[{""key_02"",475}, {""key_15"",946}, … {""key_49"",443}]"
29,16,"""insert""",16,2024-01-01,85,14452,-1333000699,6150129275156629480,-431101.53125,-9.6800e14,17507831478984.7812,2024-03-31 04:11:42.166025,2024-09-26 03:32:33.305654 CEST,"""value_719""","""1d0v2aefj8uhaef9ahsxu4wtnc9f5s…",true,"[972857599, -1176882167, … -1019948808]","{-1512636414,""s_092"",-495810.171141}","[{""key_41"",467}, {""key_06"",-589}, {""key_35"",859}]"
29,17,"""insert""",17,2024-01-01,111,3581,-504257650,-938466878178540844,799094.25,9.3039e14,-34239688416251.4062,2022-07-09 21:16:18.759884,2023-10-26 05:08:59.800859 CEST,"""value_125""","""5dmcy2ndy61aibz0vj3hii4hwgrj1p…",true,"[1342280178, -1579153467, … 1810649935]","{798106734,""s_047"",350705.188646}","[{""key_10"",-141}, {""key_23"",-532}, {""key_20"",-996}]"


## 5. Schema Evolution

ADD / RENAME / DROP columns without rewriting any Parquet files.

In [14]:
# Add a new column (metadata-only, no Parquet rewrite)
con.execute(f"ALTER TABLE {fq} ADD COLUMN priority VARCHAR DEFAULT 'normal'")
print("Column 'priority' added. Existing Parquet files untouched.")

Column 'priority' added. Existing Parquet files untouched.


In [16]:
# Verify: new column appears, old rows have the default
con.execute(f"""
    SELECT id, event_date, varchar_col, priority
    FROM {fq}
    WHERE id IN (900000001, 900000002, 1, 2, 3)
    ORDER BY id
""").pl()

id,event_date,varchar_col,priority
i64,date,str,str
1,2024-01-01,"""value_270""","""normal"""
2,2024-01-01,"""value_947""","""normal"""
3,2024-01-01,"""value_260""","""normal"""
900000001,2024-01-15,"""value_042""","""normal"""
900000002,2024-01-15,"""value_007""","""normal"""


In [17]:
# Rename column (metadata-only, zero Parquet I/O)
con.execute(f"ALTER TABLE {fq} RENAME COLUMN priority TO urgency")
print("Renamed 'priority' to 'urgency'. Zero file I/O.")

Renamed 'priority' to 'urgency'. Zero file I/O.


In [18]:
# Drop column to restore the table for the next demo
con.execute(f"ALTER TABLE {fq} DROP COLUMN urgency")
print("Dropped 'urgency'. Schema restored.")

Dropped 'urgency'. Schema restored.


In [19]:
# Check that the column has been removed
con.execute(f"""
    SELECT id, event_date, varchar_col, urgency
    FROM {fq}
    WHERE id IN (900000001, 900000002, 1, 2, 3)
    ORDER BY id
""").pl()

BinderException: Binder Error: Referenced column "urgency" not found in FROM clause!
Candidate bindings: "int8_col", "text_col", "int16_col", "int32_col", "int64_col"

LINE 2:     SELECT id, event_date, varchar_col, urgency
                                                ^

## 6. MERGE / Upsert

Atomic upsert: update existing rows + insert new ones in a single snapshot.

In [20]:
con.execute(f"""
    MERGE INTO {fq} AS target
    USING (
        VALUES
            (900000001, DATE '2024-01-15', CAST(9999 AS BIGINT),
             CAST(42.0 AS DOUBLE), 'value_042'),
            (999999999, DATE '2024-01-20', CAST(7777 AS BIGINT),
             CAST(55.5 AS DOUBLE), 'value_001')
    ) AS source(id, event_date, int64_col, float64_col, varchar_col)
    ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET int64_col = source.int64_col,
                   float64_col = source.float64_col
    WHEN NOT MATCHED THEN
        INSERT (id, event_date, int64_col, float64_col, varchar_col)
        VALUES (source.id, source.event_date, source.int64_col,
                source.float64_col, source.varchar_col)
""")
print("MERGE complete: id=900000001 updated, id=999999999 inserted.")

MERGE complete: id=900000001 updated, id=999999999 inserted.


In [21]:
# Verify MERGE results
con.execute(f"""
    SELECT id, event_date, int64_col, float64_col, varchar_col
    FROM {fq}
    WHERE id IN (900000001, 999999999)
    ORDER BY id
""").pl()

id,event_date,int64_col,float64_col,varchar_col
i64,date,i64,f64,str
900000001,2024-01-15,9999,42.0,"""value_042"""
999999999,2024-01-20,7777,55.5,"""value_001"""


## 7. Maintenance

DuckLake **does** require maintenance: file compaction, snapshot expiry, and cleanup.

In [22]:
# File statistics BEFORE compaction
con.execute(f"""
    SELECT COUNT(*)                                    AS total_files,
           ROUND(SUM(data_file_size_bytes) / 1e6, 2)  AS total_mb,
           ROUND(AVG(data_file_size_bytes) / 1e6, 2)  AS avg_file_mb,
           ROUND(MIN(data_file_size_bytes) / 1e6, 2)  AS min_file_mb,
           ROUND(MAX(data_file_size_bytes) / 1e6, 2)  AS max_file_mb
    FROM ducklake_list_files('{catalog}', '{TABLE}')
""").pl()

total_files,total_mb,avg_file_mb,min_file_mb,max_file_mb
i64,f64,f64,f64,f64
31,170.84,5.51,0.0,5.7


In [23]:
# Compact small files into larger ones
con.execute(f"CALL ducklake_merge_adjacent_files('{catalog}')")
print("ducklake_merge_adjacent_files complete.")

ducklake_merge_adjacent_files complete.


In [ ]:
# File statistics AFTER compaction
con.execute(f"""
    SELECT COUNT(*)                                    AS total_files,
           ROUND(SUM(data_file_size_bytes) / 1e6, 2)  AS total_mb,
           ROUND(AVG(data_file_size_bytes) / 1e6, 2)  AS avg_file_mb
    FROM ducklake_list_files('{catalog}', '{TABLE}')
""").pl()

In [28]:
# Expire old snapshots (aggressive for demo: 1 day)
con.execute(f"CALL ducklake_expire_snapshots('{catalog}', older_than => now() - INTERVAL '1 day')")
print("ducklake_expire_snapshots complete.")

# Clean up orphaned data files
con.execute(f"CALL ducklake_cleanup_old_files('{catalog}', cleanup_all => true)")
print("ducklake_cleanup_old_files complete.")

ducklake_expire_snapshots complete.
ducklake_cleanup_old_files complete.


In [29]:
# Final snapshot list
con.execute(f"""
    SELECT snapshot_id, snapshot_time, changes
    FROM {catalog}.snapshots()
    ORDER BY snapshot_id DESC
    LIMIT 10
""").pl()

snapshot_id,snapshot_time,changes
i64,"datetime[μs, Europe/Rome]",list[struct[2]]
34,2026-05-21 22:02:30.423307 CEST,"[{""tables_inserted_into"",[""9""]}, {""inlined_insert"",[""9""]}, {""inlined_delete"",[""9""]}]"
33,2026-05-21 21:59:08.396789 CEST,"[{""tables_altered"",[""9""]}]"
32,2026-05-21 21:59:01.058244 CEST,"[{""tables_altered"",[""9""]}]"
31,2026-05-21 21:58:21.724627 CEST,"[{""tables_altered"",[""9""]}]"


## Cleanup

Remove demo artifacts to restore the table to its pre-demo state.

In [30]:
# Uncomment and run after the presentation:
con.execute(f"DELETE FROM {fq} WHERE id IN (900000001, 900000002, 999999999)").pl()
# print("Demo rows removed.")

Count
i64
3


In [ ]:
# engine.close()